In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

In [ ]:
body = pd.read_csv('data/labeled_WM_Body_2bk_cleaned_analysis_results.csv')
tool = pd.read_csv('data/labeled_WM_Tool_2bk_cleaned_analysis_results.csv')
face = pd.read_csv('data/labeled_WM_Face_2bk_cleaned_analysis_results.csv')
place = pd.read_csv('data/labeled_WM_Place_2bk_cleaned_analysis_results.csv')
story = pd.read_csv('data/labeled_story_cleaned_analysis_results.csv')
math = pd.read_csv('data/labeled_math_cleaned_analysis_results.csv')
et_face = pd.read_csv('data/labeled_ET_Face_cleaned_analysis_results.csv')
et_shape = pd.read_csv('data/labeled_ET_Shape_cleaned_analysis_results.csv')
relational = pd.read_csv('data/labeled_relational_relational_cleaned_analysis_results.csv')


In [ ]:
SA_rank = pd.read_csv('data/SA_rank.csv')

In [ ]:
SA_rank

In [ ]:
region_to_rank = SA_rank.set_index('region')['final.rank']

In [ ]:
# List of dataframes to process
dataframes = [body, tool, face, place, story, math, et_face, et_shape, relational]

# Process each dataframe
for df in dataframes:
    # Extract the region name from the 'label' column
    df['region'] = df['label'].str.split('_').str[-1]
    
    # Map the 'final.rank' from SA_rank to the dataframe
    df['final.rank'] = df['region'].map(region_to_rank)
    
    # Drop the temporary 'region' column if not needed
    df.drop(columns=['region'], inplace=True)

In [ ]:
# face.to_csv('data/face_full_with_rank.csv', index=False)
# place.to_csv('data/place_full_with_rank.csv', index=False)

In [ ]:
relational.dropna(inplace=True)

In [ ]:
relational

***
`2025.5.18: The following cell is used for generating labelled ROI with their final.rank`

created at SFO, 10:50 PM

In [ ]:
SA_labelled = relational[['label', 'final.rank']]

In [ ]:
SA_labelled.to_csv('data/SA_labelled.csv', index=False)

***

In [ ]:
X_face = sm.add_constant(face_results['final.rank'])  # Add a constant for the intercept

``Second derivative vs. S-A rank``

In [ ]:
from scipy.stats import spearmanr

`Re-run after removing regions whose first derivative is near zero`

In [ ]:
face[abs(face['Second_Mean']) < 0.2]

In [ ]:
plt.hist(face['Second_Mean'], bins=100)

In [ ]:
# Use K-means clustering to find reasonable cutoff values for Second_Mean
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# Prepare data - use original values of Second_Mean (not absolute values)
data = face['Second_Mean'].values.reshape(-1, 1)

# Find custom initial points
min_val = np.min(face['Second_Mean'])
max_val = np.max(face['Second_Mean'])

print(f"Custom initial points:")
print(f"Minimum value: {min_val:.4f}")
print(f"Maximum value: {max_val:.4f}")

# Set custom initial centroids for 2 clusters: -1 and maximum + 1
init_centroids = np.array([[-1], [max_val + 1]])

# Method 1: K-means clustering (k=2, divide into two groups) with custom initialization
kmeans = KMeans(n_clusters=2, init=init_centroids, n_init=1, random_state=42)
clusters = kmeans.fit_predict(data)

# Get cluster centers
centers = kmeans.cluster_centers_.flatten()
print(f"Cluster centers: {centers}")

# Sort centers to identify negative and positive clusters
sorted_indices = np.argsort(centers)
negative_cluster = sorted_indices[0]
positive_cluster = sorted_indices[1]

print(f"Negative cluster center: {centers[negative_cluster]:.4f}")
print(f"Positive cluster center: {centers[positive_cluster]:.4f}")

# Calculate cutoff as midpoint between cluster centers
cutoff = (centers[negative_cluster] + centers[positive_cluster]) / 2
print(f"Cutoff: {cutoff:.4f}")

# Visualize clustering results
plt.figure(figsize=(12, 8))

# Subplot 1: Original data distribution with cluster assignments
plt.subplot(2, 2, 1)
colors = ['red', 'blue']
cluster_colors = [colors[c] for c in clusters]
plt.scatter(range(len(data)), data.flatten(), c=cluster_colors, alpha=0.6)
plt.axhline(y=cutoff, color='orange', linestyle='--', label=f'Cutoff: {cutoff:.4f}')
plt.xlabel('Data Point Index')
plt.ylabel('Second_Mean')
plt.title('K-means Clustering Results (2 clusters)')
plt.legend()

# Subplot 2: Histogram with cluster centers
plt.subplot(2, 2, 2)
plt.hist(face['Second_Mean'], bins=50, alpha=0.7, color='lightblue', edgecolor='black')
for i, center in enumerate(centers):
    plt.axvline(x=center, color=colors[i], linestyle='-', linewidth=2, 
                label=f'Cluster {i} center: {center:.4f}')
plt.axvline(x=cutoff, color='orange', linestyle='--', label=f'Cutoff: {cutoff:.4f}')
plt.xlabel('Second_Mean')
plt.ylabel('Frequency')
plt.title('Second_Mean Distribution with K-means Clusters')
plt.legend()

# Subplot 3: Data retention rate under different cutoff values
cutoff_values = np.arange(0.001, 0.1, 0.001)
retention_rates = [np.mean(np.abs(face['Second_Mean']) > cutoff_val) for cutoff_val in cutoff_values]

plt.subplot(2, 2, 3)
plt.plot(cutoff_values, retention_rates, 'b-', linewidth=2)
plt.axvline(x=abs(cutoff), color='orange', linestyle='--', label=f'|Cutoff|: {abs(cutoff):.4f}')
plt.xlabel('Cutoff Value')
plt.ylabel('Data Retention Rate')
plt.title('Data Retention Rate for Different Cutoff Values')
plt.grid(True, alpha=0.3)
plt.legend()

# Subplot 4: Elbow method to determine optimal k value
plt.subplot(2, 2, 4)
inertias = []
k_range = range(1, 8)
for k in k_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=42)
    kmeans_temp.fit(data)
    inertias.append(kmeans_temp.inertia_)

plt.plot(k_range, inertias, 'bo-')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method - Determine Optimal Number of Clusters')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Display clustering statistics
print(f"\nClustering Statistics:")
print(f"Negative cluster: {np.sum(clusters == negative_cluster)} points ({np.sum(clusters == negative_cluster)/len(clusters)*100:.1f}%)")
print(f"Positive cluster: {np.sum(clusters == positive_cluster)} points ({np.sum(clusters == positive_cluster)/len(clusters)*100:.1f}%)")


In [ ]:
face1 = face[abs(face['Second_Mean']) > 0.05]
plt.plot(face1['final.rank'], face1['Second_Mean'], 'o')

In [ ]:
face1 = face[abs(face['Second_Mean']) > .005]
# plt.plot(face1['final.rank'], face1['Second_Mean'], 'o')

# Fit the model for face1
X = sm.add_constant(face1['final.rank'])  # Add a constant for the intercept
model = sm.OLS(face1['Second_Mean'], X).fit()

# Plot the fitted line
plt.scatter(face1['final.rank'], face1['Second_Mean'])
plt.plot(face1['final.rank'], model.predict(X), label='Fitted Line', color='red')

plt.xlabel('Final Rank')
plt.ylabel('Second Mean')
plt.title('Linear Regression: Second Mean vs Final Rank (Face filtered)')
plt.legend()
plt.show()

# Output the model summary
print(model.summary())

# Calculate Spearman's rho
rho, p_value = spearmanr(face1['Second_Mean'], face1['final.rank'])
print(f"Spearman's rho: {rho}, p-value: {p_value}")

In [ ]:
face1.shape

In [ ]:
place1 = place[abs(place['Second_Mean']) > 0.005]
plt.plot(place1['final.rank'], place1['Second_Mean'], 'o')

In [ ]:
place1 = place[abs(place['Second_Mean']) > .005]
# plt.plot(place1['final.rank'], place1['Second_Mean'], 'o')

# Fit the model for place1
X = sm.add_constant(place1['final.rank'])  # Add a constant for the intercept
model = sm.OLS(place1['Second_Mean'], X).fit()

# Plot the fitted line
plt.scatter(place1['final.rank'], place1['Second_Mean'])
plt.plot(place1['final.rank'], model.predict(X), label='Fitted Line', color='red')

plt.xlabel('Final Rank')
plt.ylabel('Second Mean')
plt.title('Linear Regression: Second Mean vs Final Rank (Face filtered)')
plt.legend()
plt.show()

# Output the model summary
print(model.summary())

# Calculate Spearman's rho
rho, p_value = spearmanr(place1['Second_Mean'], place1['final.rank'])
print(f"Spearman's rho: {rho}, p-value: {p_value}")

In [ ]:
place1.shape

`Earlier content`

In [ ]:
# List of datasets to analyze
datasets = [body, tool, face, place, story, math, et_face, et_shape, relational]
dataset_names = ['Body', 'Tool', 'Face', 'Place', 'Story', 'Math', 'ET Face', 'ET Shape', 'Relational']

# Iterate through each dataset
for dataset, name in zip(datasets, dataset_names):
    # Fit the model
    X = sm.add_constant(dataset['final.rank'])  # Add a constant for the intercept
    model = sm.OLS(dataset['Second_Mean'], X).fit()

    # Plot the original data
    plt.scatter(dataset['final.rank'], dataset['Second_Mean'], label='Original Data', color='blue')

    # Plot the fitted line
    plt.plot(dataset['final.rank'], model.predict(X), label='Fitted Line', color='red')

    plt.xlabel('Final Rank')
    plt.ylabel('Second Mean')
    plt.title(f'Linear Regression: Second Mean vs Final Rank ({dataset.iloc[0]["label"].split("_")[1]})')
    plt.suptitle(f'Dataset: {name}', fontsize=14, fontweight='bold')  # Add dataset name at the top
    plt.legend()
    plt.show()

    # Output the model summary
    print(model.summary())

    # Calculate Spearman's rho
    rho, p_value = spearmanr(dataset['Second_Mean'], dataset['final.rank'])
    print(f"Spearman's rho: {rho}, p-value: {p_value}")

separate by mean first derivative sign


In [ ]:
# Iterate through each dataset
for dataset, name in zip(datasets, dataset_names):
    # Separate the dataset into two subsets based on the sign of the mean first derivative
    subset_positive = dataset[dataset['First_Mean'] > 0]
    subset_negative = dataset[dataset['First_Mean'] < 0]

    # Analyze the subset where First_Mean > 0
    if not subset_positive.empty:
        print(f"Analyzing {name} (First_Mean > 0)")
        X_positive = sm.add_constant(subset_positive['final.rank'])  # Add a constant for the intercept
        model_positive = sm.OLS(subset_positive['Second_Mean'], X_positive).fit()

        # Plot the original data
        plt.scatter(subset_positive['final.rank'], subset_positive['Second_Mean'], label='Original Data', color='blue')
        plt.plot(subset_positive['final.rank'], model_positive.predict(X_positive), label='Fitted Line', color='red')
        plt.xlabel('Final Rank')
        plt.ylabel('Second Mean')
        plt.title(f'Linear Regression: Second Mean vs Final Rank ({name}, First_Mean > 0)')
        plt.legend()
        plt.show()

        # Output the model summary
        print(model_positive.summary())

        # Calculate Spearman's rho
        rho_positive, p_value_positive = spearmanr(subset_positive['Second_Mean'], subset_positive['final.rank'])
        print(f"Spearman's rho (First_Mean > 0): {rho_positive}, p-value: {p_value_positive}")

    # Analyze the subset where First_Mean < 0
    if not subset_negative.empty:
        print(f"Analyzing {name} (First_Mean < 0)")
        X_negative = sm.add_constant(subset_negative['final.rank'])  # Add a constant for the intercept
        model_negative = sm.OLS(subset_negative['Second_Mean'], X_negative).fit()

        # Plot the original data
        plt.scatter(subset_negative['final.rank'], subset_negative['Second_Mean'], label='Original Data', color='blue')
        plt.plot(subset_negative['final.rank'], model_negative.predict(X_negative), label='Fitted Line', color='red')
        plt.xlabel('Final Rank')
        plt.ylabel('Second Mean')
        plt.title(f'Linear Regression: Second Mean vs Final Rank ({name}, First_Mean < 0)')
        plt.legend()
        plt.show()

        # Output the model summary
        print(model_negative.summary())

        # Calculate Spearman's rho
        rho_negative, p_value_negative = spearmanr(subset_negative['Second_Mean'], subset_negative['final.rank'])
        print(f"Spearman's rho (First_Mean < 0): {rho_negative}, p-value: {p_value_negative}")